# Nimble MCP: Agentic Web Search Platform

**Provider:** [Nimble](https://www.nimbleway.com/) — Real-Time Intelligence Powered by Web Search Agents

## Overview

The Nimble MCP Server gives your Databricks AI agents live access to the public web as structured data. Search across multiple engines, extract content from any URL, crawl entire sites, and run purpose-built Web Search Agents — all through standard MCP tool calls.

Every result passes through Nimble's multi-agent validation pipeline before reaching your agent. You get structured JSON, not raw text to parse.

**Key capabilities:**
- **Search** — Multi-engine web search (Google, Bing, Yandex) with focus modes for news, shopping, social, and more
- **Extract** — Pull clean content from any URL as markdown, plain text, or HTML
- **Map** — Discover every URL on a website via link-following and sitemap parsing
- **Crawl** — Multi-page crawling with path filters and progress tracking
- **Agents** — Create, run, and publish custom extraction agents for any website

**Marketplace listing:** [Install from the Databricks Marketplace](https://marketplace.databricks.com/details/1fe5e521-e9ef-49d8-97c4-b8c0a448ce15/Nimble_Nimble-MCP-Agentic-Web-Search-Platform)

**Documentation:** [docs.nimbleway.com/integrations/mcp-server/mcp-server](https://docs.nimbleway.com/integrations/mcp-server/mcp-server)

## Prerequisites

1. **Install Nimble MCP from the Databricks Marketplace** — In your workspace, go to **Marketplace > Agents > MCP Servers**, search for **Nimble**, and click **Install**.
2. **Nimble API Key** — [Sign up for free](https://online.nimbleway.com/signup) and generate a key from [Account Settings > API Keys](https://online.nimbleway.com/account-settings/api-keys). Enter this as the Bearer token during the Marketplace install.
3. **Unity Catalog Connection** — The install creates a connection named `nimble-mcp-marketplace`. Verify it under **Unity Catalog > Connections**.
4. **Databricks workspace** with the **Managed MCP Servers** preview enabled.

## Setup

Install the required packages and configure the MCP connection.

This notebook uses LangGraph with Databricks-hosted LLMs. The Nimble MCP Server is accessed through the Databricks-managed proxy via `DatabricksMCPClient`, which handles authentication automatically through your Unity Catalog connection.

In [ ]:
%pip install databricks-mcp langchain-mcp-adapters mcp databricks-langchain langgraph langchain mlflow nest_asyncio
%restart_python

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

w = WorkspaceClient()
host = w.config.host

# -------------------------------------------------------------------
# Configuration — update these to match your setup
# -------------------------------------------------------------------
NIMBLE_CONNECTION_NAME = "nimble-mcp-marketplace"  # Default name from the Marketplace install. Check yours under Unity Catalog > Connections.
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"  # Or any tool-enabled model endpoint

# Databricks-managed proxy URL for the Nimble MCP server
NIMBLE_MCP_PROXY_URL = f"{host}/api/2.0/mcp/external/{NIMBLE_CONNECTION_NAME}"

# Verify the connection works by listing available tools
mcp_client = DatabricksMCPClient(server_url=NIMBLE_MCP_PROXY_URL, workspace_client=w)
tools = mcp_client.list_tools()

print(f"Nimble MCP proxy URL: {NIMBLE_MCP_PROXY_URL}")
print(f"\nLoaded {len(tools)} Nimble tools:")
for tool in tools:
    print(f"  - {tool.name}")

### Initialize the Agent

We create a LangGraph ReAct agent backed by a Databricks-hosted LLM. We use `MultiServerMCPClient` with your Databricks workspace token to connect to the Nimble tools through the managed proxy.

In [ ]:
import os
import mlflow
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from databricks_langchain import ChatDatabricks

mlflow.langchain.autolog()

# Create a short-lived token for the MCP proxy connection
os.environ["DATABRICKS_HOST"] = w.config.host
os.environ["DATABRICKS_TOKEN"] = w.tokens.create(
    comment="nimble-mcp-demo", lifetime_seconds=1200
).token_value

llm = ChatDatabricks(endpoint=LLM_ENDPOINT)

# Connect to Nimble MCP via the Databricks-managed proxy with auth
client = MultiServerMCPClient({
    "nimble": {
        "url": NIMBLE_MCP_PROXY_URL,
        "transport": "streamable_http",
        "headers": {
            "Authorization": f"Bearer {os.environ['DATABRICKS_TOKEN']}"
        }
    }
})

langchain_tools = await client.get_tools()

# Databricks model serving rejects "additionalProperties" in tool schemas.
# LangChain's convert_to_openai_tool produces clean schemas, but
# ChatDatabricks.bind_tools() uses pydantic serialization which adds it back.
# Fix: use object.__setattr__ to bypass pydantic validation and override bind_tools.
def _strip_additional_properties(obj):
    if isinstance(obj, dict):
        obj.pop("additionalProperties", None)
        for value in obj.values():
            _strip_additional_properties(value)
    elif isinstance(obj, list):
        for item in obj:
            _strip_additional_properties(item)

clean_tool_defs = [convert_to_openai_tool(t) for t in langchain_tools]
for td in clean_tool_defs:
    _strip_additional_properties(td)

object.__setattr__(llm, 'bind_tools', lambda tools, **kw: llm.bind(tools=clean_tool_defs))

print(f"Loaded {len(langchain_tools)} tools for agent")
agent = create_react_agent(llm, langchain_tools)

---
## Use Case 1: Real-Time Web Search

Search the live web and get structured results. Nimble searches across multiple engines and returns verified data — not cached indexes from days ago.

This is useful for grounding your agents in current information, fact-checking, and research tasks.

In [ ]:
response = await agent.ainvoke({
    "messages": [{
        "role": "user",
        "content": (
            "Search for the latest news about AI agents in enterprise workflows. "
            "Summarize the top 5 results with their sources."
        )
    }]
})

print(response["messages"][-1].content)

---
## Use Case 2: Web Page Extraction

Extract structured content from any URL. The agent can pull full page content as clean markdown — ideal for feeding into RAG pipelines, building datasets, or analyzing competitor pages.

In [ ]:
response = await agent.ainvoke({
    "messages": [{
        "role": "user",
        "content": (
            "Extract the content from https://docs.nimbleway.com/integrations/mcp-server/mcp-server "
            "and list all the available MCP tools with a short description of each."
        )
    }]
})

print(response["messages"][-1].content)

---
## Use Case 3: Competitive Pricing Research

Combine search and extraction to research a real business question. Here the agent searches for product pricing data, then extracts and structures the findings — a common pattern for retail and CPG teams.

In [ ]:
response = await agent.ainvoke({
    "messages": [{
        "role": "user",
        "content": (
            "I need competitive pricing intelligence. Search for the top 5 cloud data warehouse providers "
            "and their current pricing tiers. Present the results as a comparison table "
            "with provider name, plan tiers, and starting price."
        )
    }]
})

print(response["messages"][-1].content)

---
## Use Case 4: Site Mapping & Crawling

Discover all URLs on a website, then crawl specific sections. This is useful for building comprehensive datasets, monitoring content changes, or indexing documentation sites.

In [ ]:
response = await agent.ainvoke({
    "messages": [{
        "role": "user",
        "content": (
            "Map all the URLs on docs.nimbleway.com and tell me how many pages there are. "
            "Then group them by section (e.g. SDK, integrations, web tools)."
        )
    }]
})

print(response["messages"][-1].content)

---
## Next Steps

- **Explore all 18 tools** — See the full list at [docs.nimbleway.com/integrations/mcp-server/mcp-server](https://docs.nimbleway.com/integrations/mcp-server/mcp-server)
- **Build custom extraction agents** — Use [Nimble Studio](https://online.nimbleway.com/workflow-builder) to create Web Search Agents without code, then run them via MCP
- **Add to your Databricks Assistant** — Go to Assistant Settings > MCP Servers and add your `nimble-mcp-marketplace` connection
- **Call tools directly** — Use `DatabricksMCPClient` from the `databricks-mcp` package for direct tool calls without an agent framework
- **Deploy to production** — See the [Databricks MCP deployment guide](https://docs.databricks.com/aws/en/generative-ai/mcp/external-mcp) for deploying agents with external MCP servers

**Questions?** Visit [nimbleway.com](https://www.nimbleway.com/) or reach out to [support](https://nimbleway.com/contact-general/).